In [ ]:
%matplotlib ipympl
# %matplotlib qt

import sys
import time
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import NMF, PCA
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
from ardal import Ardal
import _ardal

def makePCA(matrix, categories, D3=True, Dplus=False):
    """
    Performs PCA on a matrix and visualizes the results with a legend.

    Args:
        matrix (pd.DataFrame): The input matrix (rows are samples, columns are features).
        categories (dict): A dictionary mapping matrix indices (sample IDs) to categories.
        D3 (bool): Whether to create a 3D plot (default: True).
        Dplus (bool): Whether to use a size map (default: False).

    Raises:
        ValueError: If the matrix is not a Pandas DataFrame or if categories is not a dictionary.
    """

    # Input validation
    if not isinstance(matrix, pd.DataFrame):
        raise ValueError("matrix must be a Pandas DataFrame.")
    if not isinstance(categories, dict):
        raise ValueError("categories must be a dictionary.")

    # Color setup
    unique_categories = sorted(set(categories.values()))
    cmap = sns.color_palette("Spectral", as_cmap=True)
    colours = [cmap(i) for i in np.linspace(0, 1, len(unique_categories))]
    cdict = dict(zip(unique_categories, colours))

    matrix_indices = matrix.index
    matrix_standardized = StandardScaler().fit_transform(matrix)

    n = min(len(matrix_indices), 20)
    pca = PCA(n_components=n)
    pca_result = pca.fit_transform(matrix_standardized)
    pca_df = pd.DataFrame(data=pca_result[:, :4], columns=['PC1', 'PC2', 'PC3', 'PC4'], index=matrix_indices)

    # Plotting
    if D3:
        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection='3d')

        if Dplus:
            min_val = pca_df["PC4"].min()
            max_val = pca_df["PC4"].max()
            sizemap = 100 + ((pca_df['PC4'] - min_val) * 100) / (max_val - min_val)
            scatter = ax.scatter(pca_df['PC1'], pca_df['PC2'], pca_df['PC3'], s=sizemap, alpha=0.8)
        else:
            for category in unique_categories:
                indices = [idx for idx, cat in categories.items() if cat == category]
                subset = pca_df.loc[indices]
                scatter = ax.scatter(subset['PC1'], subset['PC2'], subset['PC3'], alpha=0.8, c=[cdict[category]] * len(subset), label=category)

            ax.legend()

        ax.set_zlabel('PC3')

    else:
        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111)

        if Dplus:
            min_val = pca_df['PC3'].min()
            max_val = pca_df['PC3'].max()
            sizemap = 200 + ((pca_df['PC3'] - min_val) * 50) / (max_val - min_val)
            scatter = ax.scatter(pca_df['PC1'], pca_df['PC2'], s=sizemap, alpha=0.7)
        else:
            for category in unique_categories:
                indices = [idx for idx, cat in categories.items() if cat == category]
                subset = pca_df.loc[indices]
                scatter = ax.scatter(subset['PC1'], subset['PC2'], alpha=0.8, c=[cdict[category]] * len(subset), label=category)

            ax.legend()

    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.grid(True)

    plt.show()



def cluster(dist_matrix, nclusters):
    from sklearn.cluster import KMeans

    dist_array = dist_matrix.values

    # Perform k-means clustering
    kmeans = KMeans(n_clusters=nclusters, random_state=0, n_init='auto')  # Set random_state for reproducibility
    cluster_labels = kmeans.fit_predict(dist_array)

    # Create the cluster membership dictionary
    cluster_membership = {}
    for i, guid in enumerate(dist_matrix.index):
        cluster_membership[guid] = cluster_labels[i]

    return cluster_membership
    

In [ ]:
from ardal import Ardal
import pandas as pd

data = ["/home/amorris/BioInf/SSD/BioInf/Plasmodium_vcfs/Pf_matrix.npy", "/home/amorris/BioInf/SSD/BioInf/Plasmodium_vcfs/Pf_headers.json"]
# data = ["/home/amorris/BioInf/burkholderia_WD/data/allele_matrices/BG_core_matrix.npy", "/home/amorris/BioInf/burkholderia_WD/data/allele_matrices/BG_core_headers.json"]
ard = Ardal(data)
# ard.stats()
df = pd.read_csv("/home/amorris/BioInf/ProtoDB/WD/Pf_SPT_hamming.csv",  index_col=0)
# meta = pd.read_csv("/home/amorris/BioInf/burkholderia_WD/data/Mullins_Onion_Jones.csv")

Loading '['/home/amorris/BioInf/SSD/BioInf/Plasmodium_vcfs/Pf_matrix.npy', '/home/amorris/BioInf/SSD/BioInf/Plasmodium_vcfs/Pf_headers.json']' as a npy/JSON pair.


In [ ]:
ardD_hamming = ard.pairwise()
meta

In [ ]:
# cat = dict(zip(list(meta["SRA"]), list(meta["Clade"])))
clus = cluster(ardD_hamming, 10)
print(clus)
makePCA(ardD_hamming, clus, D3=True)